# 02. Build Graph Snapshots

**Goal**: Implement Step 2 of the pipeline: constructing graph snapshots from node features.

For selected dates, this notebook will:
1.  Build a **Node Feature Matrix** (`x`) from price/volume/technical data.
2.  Construct an **Edge Index** (`edge_index`) based on rolling correlations between stock returns.
3.  Assemble these into **PyTorch Geometric** `Data` objects.
4.  (Optional) Save the sequence of snapshots for GNN training.

**Graph Spec**:
- **Node Ordering**: Sorted alphanumeric ticker list.
- **Node Features**: Standardized (z-score) [close, log_close, ret_1d, ret_5d, ret_20d, vol_5d, vol_20d].
- **Edges**: k-NN on 60-day absolute correlation, k=5, symmetric directed edges (i<->j).
- **Splits**: Train / Val / Test based on time range.

**Input**: `prices_df` (aligned close prices), derived features.
**Output**: List of `torch_geometric.data.Data` objects.

## 1. Imports & Setup

In [11]:
import warnings
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data

warnings.filterwarnings('ignore')

In [12]:
# Configuration
DATA_DIR = Path("../../data")
GRAPH_OUTPUT_PATH = DATA_DIR / "graphs"
GRAPH_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "vol_windows": [5, 20],
    "cov_window": 20,
    "corr_window": 60,
    "knn_k": 5,
    "min_history": 60,
    "use_abs_corr": True,
    "train_end": "2024-12-31",
    "val_end": "2025-06-30", 
}

print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {GRAPH_OUTPUT_PATH}")

Data Directory: ..\..\data
Output Directory: ..\..\data\graphs


### 1.1 Load Data

We load the historical price data. We assume either a consolidated CSV exists or we load individual ticker files from the ingestion step.

In [13]:
def load_prices(data_dir):
    """Load and align close prices from CSVs."""
    # 1. Try loading consolidated file if it exists (from previous notebook steps)
    consolidated_files = list(data_dir.glob("EGX30_OHLCV*.csv"))
    if consolidated_files:
        print(f"Loading consolidated data from {consolidated_files[0].name}...")
        df = pd.read_csv(consolidated_files[0], index_col=0, parse_dates=True)
        return df

    # 2. Fallback: Load individual files
    print("Loading individual ticker files...")
    price_frames = []
    for f in data_dir.glob("*.csv"):
        # Skip non-ticker files
        if "market_data" in f.name or "EGX30_" in f.name:
            continue
            
        try:
            df = pd.read_csv(f)
            # Standardize columns (handle 'datetime' vs 'Date' vs 'date')
            col_map = {c.lower(): c for c in df.columns}
            date_col = col_map.get('datetime', col_map.get('date'))
            close_col = col_map.get('close')
            
            if date_col and close_col:
                df = df.rename(columns={date_col: 'Date', close_col: 'Close'})
                df['Date'] = pd.to_datetime(df['Date'])
                df = df.set_index('Date')[['Close']]
                df.columns = [f.stem]  # Use filename as ticker
                price_frames.append(df)
        except Exception as e:
            print(f"Skipping {f.name}: {e}")
            
    if not price_frames:
        raise FileNotFoundError("No valid price data found in data directory.")
        
    # Align on date index
    prices = pd.concat(price_frames, axis=1).sort_index()
    prices = prices.ffill()  # Forward fill gaps
    return prices

prices_df = load_prices(DATA_DIR)
print(f"Loaded prices_df: {prices_df.shape} (Dates x Tickers)")
prices_df.tail(3)

Loading consolidated data from EGX30_OHLCV(2021-2026).csv...
Loaded prices_df: (1080, 30) (Dates x Tickers)


,ABUK,ADIB,AMOC,ARCC,BTFH,CCAP,CIEB,COMI,EAST,EGAL,...,ORAS,ORHD,ORWE,PHDC,RAYA,RMDA,SKPC,TMGH,VLMR,VLMRA
datetime,,,,,,,,,,,,,,,,,,,,,
2026-03-09 10:00:00,84.1,36.55,9.31,48.55,2.90,3.72,21.38,123.0,37.59,285.0,...,458.00,25.02,23.00,8.67,5.49,4.37,18.70,76.50,0.705,30.99
2026-03-10 10:00:00,79.9,38.97,8.57,49.25,2.96,3.61,22.19,129.5,37.85,277.0,...,464.97,25.37,23.06,8.90,5.83,4.59,18.01,79.80,0.710,30.01
2026-03-11 10:00:00,85.0,41.00,8.88,49.01,2.96,3.65,23.30,125.3,35.00,295.0,...,469.70,25.43,22.87,8.80,5.73,4.66,18.35,79.95,0.710,30.50


### 1.2 Load Pre-computed Features

Instead of re-computing, we load the feature panels created by the `Feature_Engineering.ipynb` notebook. These are stored as individual CSVs per ticker in the `C+Features` directory.

In [14]:
def load_feature_panels(features_dir: Path) -> dict[str, pd.DataFrame]:
    """
    Loads per-ticker feature CSVs and reconstructs full feature panels (feature x date x ticker).
    """
    feature_files = list(features_dir.glob("*_Features.csv"))
    if not feature_files:
        raise FileNotFoundError(f"No feature files found in {features_dir}. Please run the feature engineering notebook first.")

    all_series = {}
    for f_path in feature_files:
        ticker = f_path.stem.replace("_Features", "")
        ticker_df = pd.read_csv(f_path, index_col="datetime", parse_dates=True)
        for feature_name in ticker_df.columns:
            if feature_name not in all_series:
                all_series[feature_name] = []
            
            series = ticker_df[feature_name]
            series.name = ticker
            all_series[feature_name].append(series)

    panels = {name: pd.concat(series_list, axis=1).sort_index() for name, series_list in all_series.items()}
    print(f"Loaded {len(panels)} feature panels for {len(feature_files)} tickers.")
    return panels

# Load all panels from the C+Features directory
FEATURES_DIR = DATA_DIR / "C+Features"
feature_panels = load_feature_panels(FEATURES_DIR)

# --- Schema & Alignment Checks ---
expected_features = {"close", "log_close", "ret_1d", "ret_5d", "ret_20d", "vol_5d", "vol_20d"}
missing = expected_features - set(feature_panels.keys())
assert not missing, f"Missing required features in C+Features: {missing}"

# Enforce sorted tickers and common index
tickers = sorted(feature_panels["close"].columns)
common_index = feature_panels["close"].index

for name, panel in feature_panels.items():
    # Subset columns to sorted tickers
    panel = panel[tickers]
    feature_panels[name] = panel
    # Intersect dates
    common_index = common_index.intersection(panel.index)

# Reindex all
for name, panel in feature_panels.items():
    feature_panels[name] = panel.loc[common_index]

# Re-create the data structures expected by the notebook
prices_df = feature_panels['close']
# The 'ret_1d' from Feature_Engineering notebook is log returns, which is suitable for correlation.
returns_df = feature_panels['ret_1d']

volatility_features = {
    'vol_5d': feature_panels['vol_5d'],
    'vol_20d': feature_panels['vol_20d'],
}

# Note: The features from the file use log returns for momentum
momentum_features = {
    'ret_1d': feature_panels['ret_1d'],
    'ret_5d': feature_panels['ret_5d'],
    'ret_20d': feature_panels['ret_20d'],
}

# Load Normalization Stats
stats_path = DATA_DIR / "feature_stats.json"
with open(stats_path, "r") as f:
    stats = json.load(f)
feature_means = stats["means"]
feature_stds = stats["stds"]

print("Feature dictionaries built from loaded files.")

Loaded 7 feature panels for 30 tickers.
Feature dictionaries built from loaded files.


In [15]:
def build_node_features(
    as_of_date,
    prices: pd.DataFrame,
    returns: pd.DataFrame,
    volatility_dict: dict,
    momentum_dict: dict | None = None,
    feature_means: dict = feature_means,
    feature_stds: dict = feature_stds,
) -> pd.DataFrame:
    """
    Build a node-feature matrix for all tickers at a single date.

    Parameters
    ----------
    as_of_date : str or pd.Timestamp
        Date at which to extract node features.
    prices : pd.DataFrame
        Aligned close-price panel of shape [num_days, num_stocks].
    returns : pd.DataFrame
        Daily return panel aligned with prices.
    volatility_dict : dict[str, pd.DataFrame]
        Mapping from feature name to panel, e.g. {"vol_5d": df, "vol_20d": df}.
    momentum_dict : dict[str, pd.DataFrame] | None
        Optional mapping of return-based features such as ret_5d, ret_20d.
    include_price_level : bool
        If True, include close price and log-close in the feature matrix.
    dropna : bool
        If True, drop rows containing missing values.

    Returns
    -------
    pd.DataFrame
        Node features indexed by ticker, shape [num_stocks, num_features].
    """
    as_of_date = pd.Timestamp(as_of_date)
    
    if as_of_date not in prices.index:
        raise KeyError(f"Date {as_of_date} not found in aligned price panel.")

    # Check insufficient history check
    idx = prices.index.get_loc(as_of_date)
    if idx < CONFIG["min_history"]:
        raise ValueError(f"Not enough history before {as_of_date} for full feature set.")
    
    features = pd.DataFrame(index=prices.columns)
    features.index.name = "ticker"

    # Standardize Features using loaded stats
    # Price Level
    features["close"] = (prices.loc[as_of_date] - feature_means["close"]) / feature_stds["close"]
    features["log_close"] = (np.log(prices.loc[as_of_date]) - feature_means["log_close"]) / feature_stds["log_close"]

    # Return
    features["ret_1d"] = (returns.loc[as_of_date] - feature_means["ret_1d"]) / feature_stds["ret_1d"]

    # Volatility
    for name, panel in volatility_dict.items():
        features[name] = (panel.loc[as_of_date] - feature_means[name]) / feature_stds[name]

    # Momentum
    if momentum_dict:
        for name, panel in momentum_dict.items():
            if name == "ret_1d": continue
            features[name] = (panel.loc[as_of_date] - feature_means[name]) / feature_stds[name]
    
    # Fill any remaining NaNs with 0 (mean) after standardization
    features = features.fillna(0.0)

    return features.sort_index()

## 2. Correlation-Based Edge Construction

We define how to connect stocks based on their historical correlation over a lookback window.

In [16]:
def build_edge_index_from_corr(
    returns_df: pd.DataFrame,
    as_of_date: pd.Timestamp,
    window: int = 60,
    k: int = 5,
    method: str = "knn",
    corr_threshold: float = 0.5,
    use_abs_corr: bool = True,
) -> torch.Tensor:
    """
    Build a correlation-based edge_index for PyTorch Geometric.
    """
    ts = pd.Timestamp(as_of_date)
    
    # 1. Validation & Slicing
    if ts not in returns_df.index:
        raise ValueError(f"Date {ts} not in returns index")
        
    # Get integer location of the date
    idx = returns_df.index.get_loc(ts)
    
    if idx < window:
        raise ValueError(f"Not enough history for window={window} at {ts}")
        
    # Slice: (t - window) to t (inclusive)
    # returns_df is sorted by date
    window_slice = returns_df.iloc[idx - window + 1 : idx + 1]
    
    # 2. Compute Correlation Matrix
    # shape: [num_tickers, num_tickers]
    corr_matrix = window_slice.corr().fillna(0.0)
    matrix_vals = corr_matrix.values
    if use_abs_corr:
        matrix_vals = np.abs(matrix_vals)

    num_nodes = len(tickers)
    edge_set = set()
    
    if method == "knn":
        # For each node, find top k correlated neighbors (excluding self)
        # We fill diagonal with -infinity so self is not selected
        np.fill_diagonal(matrix_vals, -np.inf)
        
        for i in range(num_nodes):
            # Get indices of top k values in row i
            # argsort returns ascending, so take last k
            # abs correlation? usually we want positive correlation for "similarity"
            # lets assume raw correlation for now (or abs if specified)
            neighbors = np.argsort(matrix_vals[i])[-k:]
            
            for neighbor_idx in neighbors:
                if i == neighbor_idx:
                    continue
                # Undirected/Symmetric logic: add both directions to set to avoid dups
                # But since we iterate all 'i', if we just add (i, j) and (j, i) we might get duplicates
                # if reciprocity isn't perfect. 
                # Safer to just add i->j and j->i to a set of tuples.
                edge_set.add((i, neighbor_idx))
                edge_set.add((neighbor_idx, i))

    elif method == "threshold":
        # Edges where corr > threshold
        rows, cols = np.where(matrix_vals > corr_threshold)
        for r, c in zip(rows, cols):
            if r != c:  # exclude self loops
                edge_set.add((r, c))
                edge_set.add((c, r))
    else:
        raise ValueError(f"Unknown method: {method}")
        
    # 4. Convert to Tensor
    if edge_set:
        sources, targets = zip(*edge_set)
    else:
        sources, targets = [], []

    edge_index = torch.tensor([sources, targets], dtype=torch.long)
    
    return edge_index

## 3. Build Graph Snapshot

Combine node features and edges into a single object.

In [17]:
def build_graph_snapshot(
    as_of_date,
    prices_df: pd.DataFrame,
    returns_df: pd.DataFrame,
    volatility_dict: dict[str, pd.DataFrame],
    momentum_dict: dict[str, pd.DataFrame],
    window_corr: int = 60,
    k: int = 5,
    use_abs_corr: bool = True,
) -> Data:
    """
    Build a single graph snapshot for a given date.
    """
    date_ts = pd.Timestamp(as_of_date)
    
    # 1. Build Node Features (X)
    # Use the function from Feature_Engineering, ensuring we keep all nodes.
    features_df = build_node_features(
        as_of_date=date_ts,
        prices=prices_df,
        returns=returns_df,
        volatility_dict=volatility_dict,
        momentum_dict=momentum_dict,
    )
    
    # Convert to Tensor [num_nodes, num_features]
    x = torch.tensor(features_df.values, dtype=torch.float)
    
    # 2. Build Edge Index
    edge_index = build_edge_index_from_corr(
        returns_df=returns_df,
        as_of_date=date_ts,
        window=window_corr,
        k=k,
        method="knn",
        use_abs_corr=use_abs_corr
    )
    
    # 3. Create Data Object
    data = Data(x=x, edge_index=edge_index)
    
    # Metadata (Optional but useful)
    data.date = str(date_ts.date())
    data.tickers = list(features_df.index)
    data.num_nodes = len(features_df)
    
    # Optional Diagnostics
    deg = torch.bincount(edge_index[0], minlength=x.size(0))
    data.degrees = deg

    return data

## 4. Build Multiple Snapshots

We generate snapshots for a range of dates to simulate a dataset.

In [18]:
def build_snapshots_for_dates(date_index, label):
    snapshots = []
    errors = []
    print(f"Building {label} snapshots ({len(date_index)} days)...")
    
    for dt in date_index:
        try:
            g = build_graph_snapshot(
                as_of_date=dt,
                prices_df=prices_df,
                returns_df=returns_df,
                volatility_dict=volatility_features,
                momentum_dict=momentum_features,
                window_corr=CONFIG["corr_window"],
                k=CONFIG["knn_k"],
                use_abs_corr=CONFIG["use_abs_corr"]
            )
            snapshots.append(g)
        except Exception as e:
            errors.append((dt, str(e)))
            
    print(f"{label}: built {len(snapshots)} graphs, {len(errors)} errors.")
    if errors:
        print(f"  First error: {errors[0][1]}")
    return snapshots

# Define Splits based on dates
all_dates = prices_df.index
train_end = pd.Timestamp(CONFIG["train_end"])
val_end = pd.Timestamp(CONFIG["val_end"])

train_dates = all_dates[all_dates <= train_end]
val_dates = all_dates[(all_dates > train_end) & (all_dates <= val_end)]
test_dates = all_dates[all_dates > val_end]

train_graphs = build_snapshots_for_dates(train_dates, "Train")
val_graphs = build_snapshots_for_dates(val_dates, "Val")
test_graphs = build_snapshots_for_dates(test_dates, "Test")

# For compatibility with the sanity check cell below
graph_snapshots = train_graphs + val_graphs + test_graphs

Building Train snapshots (771 days)...
Train: built 711 graphs, 60 errors.
  First error: Not enough history before 2021-10-18 11:00:00 for full feature set.
Building Val snapshots (115 days)...
Val: built 115 graphs, 0 errors.
Building Test snapshots (174 days)...
Test: built 174 graphs, 0 errors.


## 5. Sanity Checks & Saving

In [19]:
# Inspect the last graph
if graph_snapshots:
    g = graph_snapshots[-1]
    print("Sample Graph Snapshot (Latest):")
    print(g)
    print(f"  - Num Nodes: {g.num_nodes}")
    print(f"  - Node Feature Shape: {g.x.shape}")
    print(f"  - Edge Index Shape: {g.edge_index.shape}")
    print(f"  - Tickers (first 5): {g.tickers[:5]}")

    deg = g.degrees.float()
    print(f"  - Degree Stats: Min={deg.min():.0f}, Mean={deg.mean():.2f}, Max={deg.max():.0f}")
    
    # Assertions
    assert g.x.shape[0] == prices_df.shape[1], "Node count mismatch"
    assert g.edge_index.shape[0] == 2, "Edge index invalid shape"
    assert not torch.isnan(g.x).any(), "NaNs found in node features"
    
    print("\nSanity checks passed.")

Sample Graph Snapshot (Latest):
Data(x=[30, 7], edge_index=[2, 204], date='2026-03-11', tickers=[30], num_nodes=30, degrees=[30])
  - Num Nodes: 30
  - Node Feature Shape: torch.Size([30, 7])
  - Edge Index Shape: torch.Size([2, 204])
  - Tickers (first 5): ['ABUK', 'ADIB', 'AMOC', 'ARCC', 'BTFH']
  - Degree Stats: Min=5, Mean=6.80, Max=14

Sanity checks passed.


In [20]:
# Save to disk
torch.save(train_graphs, GRAPH_OUTPUT_PATH / "EGX30_Graphs_train.pt")
torch.save(val_graphs, GRAPH_OUTPUT_PATH / "EGX30_Graphs_val.pt")
torch.save(test_graphs, GRAPH_OUTPUT_PATH / "EGX30_Graphs_test.pt")
print("Saved Train/Val/Test graph snapshots to disk.")

Saved Train/Val/Test graph snapshots to disk.
